In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path(r"D:\GeneVISTA")
processed_folder = PROJECT_ROOT / "data" / "processed"

evaluation_bundle = Path(
    r"D:\GeneVISTA\models\temporal_v1\20260913_025350_159532"
)

evaluation_plan = json.loads(
    (evaluation_bundle / "evaluation_plan.json").read_text(
        encoding="utf-8"
    )
)

assert sklearn.__version__ == evaluation_plan["sklearn_version"], (
    "The scikit-learn version differs from the training environment."
)

feature_config = evaluation_plan["feature_config"]
categorical_features = feature_config["categorical_features"]
numeric_features = feature_config["numeric_features"]
model_features = categorical_features + numeric_features
join_keys = feature_config["join_keys"]

test_files = evaluation_plan["test_files"]

baseline = pd.read_csv(
    processed_folder / "model_features" / test_files["baseline_features"],
    dtype={key: "string" for key in join_keys}
)

history = pd.read_csv(
    processed_folder / "temporal_features_v1" / test_files["temporal_features"],
    dtype={key: "string" for key in join_keys}
)

labels = pd.read_csv(
    processed_folder / "model_labels" / test_files["labels"],
    dtype="string"
)

for table in [baseline, history, labels]:
    assert len(table) == evaluation_plan["expected_test_rows"]
    assert not table.duplicated(join_keys).any()
    assert table["feature_snapshot"].eq("2025-01").all()

assert labels["split"].eq("test").all()
assert labels["outcome_snapshot"].eq("2026-01").all()
assert labels["target"].isin(feature_config["target_classes"]).all()

test_data = baseline.merge(
    history,
    on=join_keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert test_data["_merge"].eq("both").all()
test_data = test_data.drop(columns="_merge")

test_data = test_data.merge(
    labels[join_keys + ["target"]],
    on=join_keys,
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert test_data["_merge"].eq("both").all()

# Preserve identifiers for traceable evaluation records.
test_ids = test_data[join_keys].copy()
X_test = test_data[model_features].copy()
y_test = test_data["target"].copy()

for column in numeric_features:
    X_test[column] = pd.to_numeric(
        X_test[column], errors="raise"
    ).astype("float64")

    assert not np.isinf(X_test[column].to_numpy()).any()

for column in categorical_features:
    X_test[column] = X_test[column].astype(object)
    X_test[column] = X_test[column].where(
        X_test[column].notna(), np.nan
    )

assert not {
    "VariationID", "feature_snapshot", "outcome_snapshot", "split", "target"
} & set(X_test.columns)

frozen_models = {}

for name in [
    evaluation_plan["primary_model"],
    evaluation_plan["comparator_model"]
]:
    frozen_models[name] = joblib.load(
        evaluation_bundle / f"{name}.joblib"
    )

    assert list(frozen_models[name].feature_names_in_) == model_features

del baseline, history, labels, test_data

print("HELD-OUT TEST SETUP VERIFIED")
print(f"Test interval: {evaluation_plan['test_interval']}")
print(f"Examples: {len(X_test):,}")
print(f"Features: {X_test.shape[1]}")
print("Loaded models:", ", ".join(frozen_models))
print("No fitting or predictions performed.")

HELD-OUT TEST SETUP VERIFIED
Test interval: 2025-01_to_2026-01
Examples: 1,456,851
Features: 12
Loaded models: primary_gradient_boosting, comparator_balanced_logistic
No fitting or predictions performed.


In [2]:
from datetime import datetime
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    average_precision_score
)

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
results_folder = evaluation_bundle / f"test_evaluation_{run_name}"
results_folder.mkdir(parents=True, exist_ok=False)

class_names = feature_config["target_classes"]
ranking_targets = ["became_benign", "became_pathogenic"]

metric_rows = []
ranking_rows = []
review_rows = []

tie_order = np.random.default_rng(
    evaluation_plan["tie_break_seed"]
).permutation(len(y_test))

# Fixed reference: the majority class learned from training was stayed_vus.
majority_predictions = np.full(len(y_test), "stayed_vus")

metric_rows.append({
    "model": "majority_class_reference",
    "accuracy": accuracy_score(y_test, majority_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, majority_predictions
    ),
    "macro_f1": f1_score(
        y_test,
        majority_predictions,
        labels=class_names,
        average="macro",
        zero_division=0
    )
})

for name, model in frozen_models.items():
    print(f"Evaluating {name}...")

    probabilities = model.predict_proba(X_test)
    model_classes = list(model.named_steps["classifier"].classes_)

    assert probabilities.shape == (len(y_test), len(class_names))
    assert set(model_classes) == set(class_names)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()
    np.testing.assert_allclose(
        probabilities.sum(axis=1), 1.0, atol=1e-6
    )

    predictions = np.asarray(model_classes)[
        probabilities.argmax(axis=1)
    ]

    metric_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "macro_f1": f1_score(
            y_test,
            predictions,
            labels=class_names,
            average="macro",
            zero_division=0
        )
    })

    report = classification_report(
        y_test,
        predictions,
        labels=class_names,
        output_dict=True,
        zero_division=0
    )

    pd.DataFrame(report).T.to_csv(
        results_folder / f"{name}_classification_report.csv"
    )

    matrix = pd.DataFrame(
        confusion_matrix(y_test, predictions, labels=class_names),
        index=pd.Index(class_names, name="actual"),
        columns=pd.Index(class_names, name="predicted")
    )

    assert int(matrix.to_numpy().sum()) == len(y_test)
    matrix.to_csv(results_folder / f"{name}_confusion_matrix.csv")

    print(
        classification_report(
            y_test,
            predictions,
            labels=class_names,
            digits=4,
            zero_division=0
        )
    )

    for target in ranking_targets:
        actual = y_test.eq(target).to_numpy()
        total_positives = int(actual.sum())
        assert total_positives > 0

        scores = probabilities[:, model_classes.index(target)]
        prevalence = float(actual.mean())
        ap = average_precision_score(actual, scores)

        ranking_rows.append({
            "model": name,
            "target": target,
            "prevalence": prevalence,
            "average_precision": ap,
            "AP_divided_by_prevalence": ap / prevalence
        })

        ranked = tie_order[
            np.argsort(-scores[tie_order], kind="stable")
        ]

        for requested_size in evaluation_plan["review_sizes"]:
            size = min(requested_size, len(y_test))
            selected = ranked[:size]
            correct = int(actual[selected].sum())

            review_rows.append({
                "model": name,
                "target": target,
                "review_size": size,
                "true_reclassifications": correct,
                "other_outcomes": size - correct,
                "precision_percent": 100 * correct / size,
                "recall_percent": 100 * correct / total_positives,
                "lift_over_prevalence": (correct / size) / prevalence
            })

    # Save traceable scores for later auditing.
    prediction_table = test_ids.reset_index(drop=True).copy()
    prediction_table["actual_target"] = y_test.to_numpy()
    prediction_table["predicted_target"] = predictions

    for column, target in enumerate(model_classes):
        prediction_table[f"score_{target}"] = probabilities[:, column]

    prediction_table.to_csv(
        results_folder / f"{name}_test_predictions.csv.gz",
        index=False,
        compression="gzip",
        chunksize=100_000
    )

    del prediction_table, probabilities, predictions

test_metrics = pd.DataFrame(metric_rows)
test_ranking = pd.DataFrame(ranking_rows)
test_review_lists = pd.DataFrame(review_rows)

primary_metric = (
    test_ranking.groupby("model", as_index=False)["average_precision"]
    .mean()
    .rename(columns={
        "average_precision": "mean_reclassification_average_precision"
    })
)

for filename, table in {
    "test_metrics.csv": test_metrics,
    "test_ranking.csv": test_ranking,
    "test_review_lists.csv": test_review_lists,
    "primary_metric.csv": primary_metric
}.items():
    table.to_csv(results_folder / filename, index=False)

evaluation_record = {
    "evaluation_plan": evaluation_plan,
    "test_rows": len(y_test),
    "models_evaluated": list(frozen_models),
    "test_evaluated": True,
    "fitting_performed": False,
    "completed_at": datetime.now().isoformat()
}

(results_folder / "evaluation_record.json").write_text(
    json.dumps(evaluation_record, indent=2),
    encoding="utf-8"
)

print("\nHELD-OUT EVALUATION COMPLETE")
print(f"Results saved to: {results_folder}")

print("\nCLASSIFICATION METRICS")
display(test_metrics)

print("\nPRIMARY METRIC")
display(primary_metric)

print("\nRANKING QUALITY")
display(test_ranking)

print("\nREVIEW-LIST RESULTS")
print(test_review_lists.round(4).to_string(index=False))

Evaluating primary_gradient_boosting...
                   precision    recall  f1-score   support

       stayed_vus     0.9937    1.0000    0.9968   1447648
    became_benign     0.0000    0.0000    0.0000      7098
became_pathogenic     0.0000    0.0000    0.0000      2105

         accuracy                         0.9937   1456851
        macro avg     0.3312    0.3333    0.3323   1456851
     weighted avg     0.9874    0.9937    0.9905   1456851

Evaluating comparator_balanced_logistic...
                   precision    recall  f1-score   support

       stayed_vus     0.9934    0.2921    0.4515   1447648
    became_benign     0.0044    0.6037    0.0088      7098
became_pathogenic     0.0133    0.3753    0.0257      2105

         accuracy                         0.2937   1456851
        macro avg     0.3370    0.4237    0.1620   1456851
     weighted avg     0.9872    0.2937    0.4487   1456851


HELD-OUT EVALUATION COMPLETE
Results saved to: D:\GeneVISTA\models\temporal_v1\20260

,model,accuracy,balanced_accuracy,macro_f1
0,majority_class_reference,0.993683,0.333333,0.332277
1,primary_gradient_boosting,0.993683,0.333333,0.332277
2,comparator_balanced_logistic,0.293747,0.423699,0.161974



PRIMARY METRIC


,model,mean_reclassification_average_precision
0,comparator_balanced_logistic,0.007135
1,primary_gradient_boosting,0.007183



RANKING QUALITY


,model,target,prevalence,average_precision,AP_divided_by_prevalence
0,primary_gradient_boosting,became_benign,0.004872,0.006172,1.266793
1,primary_gradient_boosting,became_pathogenic,0.001445,0.008194,5.670974
2,comparator_balanced_logistic,became_benign,0.004872,0.004087,0.838751
3,comparator_balanced_logistic,became_pathogenic,0.001445,0.010184,7.048112



REVIEW-LIST RESULTS
                       model            target  review_size  true_reclassifications  other_outcomes  precision_percent  recall_percent  lift_over_prevalence
   primary_gradient_boosting     became_benign          100                       3              97               3.00          0.0423                6.1574
   primary_gradient_boosting     became_benign         1000                      14             986               1.40          0.1972                2.8735
   primary_gradient_boosting     became_benign         5000                      58            4942               1.16          0.8171                2.3809
   primary_gradient_boosting became_pathogenic          100                       5              95               5.00          0.2375               34.6045
   primary_gradient_boosting became_pathogenic         1000                      23             977               2.30          1.0926               15.9181
   primary_gradient_boosting became_p